# Week 2. Counting Is Already a Model

**What you'll do today.** Build a word-counter by hand-logic and watch tokenizing,
stemming, and stop lists change its answer. Count phrases: n-grams across three corpora,
then the week's featured trial (Bollen v. Schmidt) with its own data, pulled live from
Google Books. Pick up the basic statistics of counts: distributions, rates, and what a
mean hides. Reweight with tf-idf and watch k-means separate three books from counts
alone. Upgrade the signature-vocabulary trick into keyness, and finish with the one
statistics move the course requires: the shuffle test for whether a difference is real.
(You counted an image pixel by pixel in Week 1; this week goes deep on words.)


> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Fetch the pinned package lists if they aren't beside the notebook (this happens
# whenever you open a single notebook straight from GitHub in Colab).
import os, urllib.request
_RAW = "https://raw.githubusercontent.com/lucianli123/culture-as-data-2026/main/notebooks/"
for _f in ("requirements.txt", "constraints.txt"):
    if not os.path.exists(_f):
        urllib.request.urlretrieve(_RAW + _f, _f)

# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib pillow

In [ ]:
# Imports. If this cell fails, it almost always means the install cell above didn't run
# this session, re-run it, then re-run this. (See the cheat sheet, entry 5.)
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
print("imports ok")

In [ ]:
import os

## Three real corpora

A *corpus* is just "a pile of text you're studying." Today's three are real and
pulled live from Project Gutenberg: Shakespeare's 154 sonnets, and two novels, *Frankenstein*
(1818) and *Dracula* (1897), split into paragraphs. Same counter, three very different
voices.

In [ ]:
import re

def load_gutenberg(url):
    """Fetch from Project Gutenberg and strip the boilerplate."""
    import requests
    raw = requests.get(url, timeout=30).text.replace("\r\n", "\n")
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]

def paragraphs(text, min_chars=200):
    return [p.strip().replace("\n", " ") for p in text.split("\n\n") if len(p.strip()) >= min_chars]

_s = load_gutenberg("https://www.gutenberg.org/cache/epub/1041/pg1041.txt")
_parts = re.split(r"\n\s*([IVXLC]+)\s*\n", _s)
poems = [_parts[i+1].strip() for i in range(1, len(_parts) - 1, 2) if 200 < len(_parts[i+1].strip()) < 900]
frank = paragraphs(load_gutenberg("https://www.gutenberg.org/cache/epub/84/pg84.txt"))
drac = paragraphs(load_gutenberg("https://www.gutenberg.org/cache/epub/345/pg345.txt"))
corpora = {"sonnets": poems, "frankenstein": frank, "dracula": drac}
print(len(poems), "sonnets |", len(frank), "Frankenstein paragraphs |", len(drac), "Dracula paragraphs")

## Bag-of-words by hand: counting *requires defining*

Before any library, count by hand-logic.

In [ ]:
def tokenize(text, fold_case=True, strip_punct=True):
    if fold_case:
        text = text.lower()
    if strip_punct:
        text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if t]

sample_text = " ".join(poems[:20])
counts = Counter(tokenize(sample_text))
print("Top words in the first twenty sonnets:")
for word, n in counts.most_common(8):
    print(f"  {word:8s} {n}")

In [ ]:
# A "stop word" list is itself a choice. Watch the top-words list change when we drop
# common function words, the same kind of decision as merging run/running.
STOP = {"i","we","you","it","the","a","and","to","of","in","so","be","on","my","me","now"}
counts_nostop = Counter(t for t in tokenize(sample_text) if t not in STOP)
print("Top words with stop-words removed:")
for word, n in counts_nostop.most_common(8):
    print(f"  {word:8s} {n}")

### What counts as a word? Tokens, not words

Models never see words.

## Stemming, lemmatization, and a real stop list

`tokenize` split the text. It did not decide that "love", "loves", and "loving" are the
same word. Two standard tools make that call, and they disagree. A **stemmer** chops
endings by rule. A **lemmatizer** looks the word up and returns its dictionary form.
NLTK ships both, plus a stop list far longer than our hand-built one. (Colab has NLTK
preinstalled; elsewhere, `pip install nltk` first.)


In [ ]:
# needs: pip install nltk  (already on Colab)
try:
    import nltk
    for pkg in ["stopwords", "wordnet", "omw-1.4"]:
        nltk.download(pkg, quiet=True)
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer, WordNetLemmatizer

    NLTK_STOP = set(stopwords.words("english"))
    print(f"NLTK's English stop list: {len(NLTK_STOP)} words. Our hand list: {len(STOP)}.")
    print("in NLTK's but not ours:", sorted(NLTK_STOP - STOP)[:10], "...")
    print("in neither, though the sonnets need them:",
          [w for w in ["thy", "thou", "thee", "doth", "hath"] if w not in NLTK_STOP])
    # Even a list that long misses this corpus's function words. No stop list is
    # neutral, and none is finished: it must fit the corpus in front of you.

    stem = PorterStemmer().stem
    lemma = WordNetLemmatizer().lemmatize
    forms = ["loves", "loving", "loved", "lovely", "beauty", "beauties", "eyes", "was"]
    print(f"\n{'word':10s} {'stem':10s} {'lemma (as noun)':16s} {'lemma (as verb)':16s}")
    for w in forms:
        print(f"{w:10s} {stem(w):10s} {lemma(w):16s} {lemma(w, 'v'):16s}")
    # The stemmer is fast and crude: "beauti" is not a word. The lemmatizer is
    # accurate but needs the part of speech: told nothing, it reads "was" as a noun
    # and returns "wa". Both are choices about what counts as the same word.

    tokens = tokenize(" ".join(poems))
    print(f"\nsonnet vocabulary: {len(set(tokens))} raw, "
          f"{len({stem(t) for t in tokens})} stemmed, "
          f"{len({lemma(t) for t in tokens})} lemmatized")
    # Merging forms shrinks the vocabulary and fattens each count. Whether run and
    # running are one word or two is your call, and it changes every chart downstream.
except Exception as e:
    print("NLTK unavailable here, skipping this demo:", e)


In [ ]:
# spaCy tags and lemmatizes in one pass, so it gets "loving" right without being told.
# Preinstalled on Colab; heavier elsewhere (pip install spacy, then download the model).
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    for t in nlp("Shall I compare thee to a summer's day?"):
        print(f"{t.text:10s} {t.pos_:6s} {t.lemma_}")
except Exception as e:
    print("spaCy not available here; the NLTK cell above covers the lesson.", e)


## Zipf's law: every corpus has the same shape

Rank the words by frequency and plot count against rank, both on log scales. Any corpus
in any language gives roughly a straight line: the second word is about half as common
as the first, the tenth about a tenth as common. A handful of words does most of the
work, and thousands appear once.


In [ ]:
plt.figure(figsize=(7, 4))
top_count = 0
for name, docs_ in corpora.items():
    freqs = sorted(Counter(tokenize(" ".join(docs_))).values(), reverse=True)
    top_count = max(top_count, freqs[0])
    plt.loglog(range(1, len(freqs) + 1), freqs, label=f"{name} ({len(freqs):,} distinct words)")
ranks = np.arange(1, 10_000)
plt.loglog(ranks, top_count / ranks, "k--", linewidth=1, label="1/rank reference")
plt.xlabel("word rank"); plt.ylabel("count"); plt.legend()
plt.title("Zipf's law: three corpora, one shape")
plt.tight_layout(); plt.show()

# The head of the curve is the stop list: the, and, of are so common they swamp any
# count that keeps them. The tail is the other warning: most words are too rare to
# count reliably, which is part of why Week 5 replaces words with dimensions.


## N-grams at corpus scale

Week 1 counted word pairs in the sonnets. The same move, across all three corpora: the
top bigrams are each book's fingerprint.


In [ ]:
def bigram_counts(docs_, stop=()):
    c = Counter()
    for d in docs_:
        ws = tokenize(d)
        for a, b in zip(ws, ws[1:]):
            if a not in stop and b not in stop:
                c[a + " " + b] += 1
    return c

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (name, docs_) in zip(axes, corpora.items()):
    top = bigram_counts(docs_, STOP).most_common(8)
    ax.barh([p for p, _ in top][::-1], [n for _, n in top][::-1], color="#3f6f5f")
    ax.set_title(name, fontsize=10)
fig.suptitle("Top bigrams per corpus (stop words filtered)", y=1.04)
plt.tight_layout(); plt.show()
# The sonnets' pairs are thy/thou-heavy because our stop list is modern: the Week 1
# lesson again, at phrase scale. The novels' pairs are already almost characters.


## The trial, live: Bollen v. Schmidt

The week's featured exchange is an n-gram study. Bollen et al. (PNAS 2021) counted
cognitive-distortion phrases ("I am a failure") in Google Books and found a hockey
stick after 2000; Schmidt et al. replied that the corpus itself changed, with more
fiction scanned in recent years. The Ngram Viewer has a JSON endpoint, so the evidence
is one request away. This is a probe, not a replication: the paper counts 241 phrases,
we pull two. (If you get HTTP 429, the endpoint is rate-limited; wait a minute.)


In [ ]:
import requests

def google_ngrams(phrases, corpus="en-2019", start=1900, end=2019):
    """Yearly share of each phrase in Google Books, straight from the Ngram Viewer."""
    r = requests.get("https://books.google.com/ngrams/json",
                     params={"content": ",".join(phrases), "year_start": start,
                             "year_end": end, "corpus": corpus, "smoothing": 3},
                     timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    return pd.DataFrame({s["ngram"]: s["timeseries"] for s in r.json()},
                        index=range(start, end + 1))

claim = google_ngrams(["I am a failure", "everyone hates me"])
claim.plot(figsize=(8, 3), color=["#7a3b2e", "#b9852f"])
plt.title("The claim, reproduced live: distortion phrases in Google Books")
plt.ylabel("share of all bigrams/n-grams"); plt.tight_layout(); plt.show()
# There is the hockey stick. Now the objection.


In [ ]:
# The objection, counted. If the corpus quietly gained fiction, phrases that mark
# fiction should hockey-stick too, with no change in anyone's mind:
suspects = google_ngrams(["she whispered", "he grinned"])
suspects.plot(figsize=(8, 3), color=["#3f6f5f", "#2e5f8a"])
plt.title("Fiction markers rise the same way: the corpus changed under the count")
plt.tight_layout(); plt.show()

# The sharpest single check: the same phrase inside fiction only, against all books.
both = pd.DataFrame({
    "all books": google_ngrams(["I am a failure"]).iloc[:, 0],
    "fiction only": google_ngrams(["I am a failure"], corpus="en-fiction-2019").iloc[:, 0],
})
both.plot(figsize=(8, 3), color=["#7a3b2e", "#6d4b7c"])
plt.title('"I am a failure": flat within fiction, rising in the mixed corpus')
plt.tight_layout(); plt.show()
# Within fiction the phrase is roughly level for a century; the rise lives in the mixed
# corpus, which is what you would see if composition did the work. The authors' rebuttal
# removed fiction and reported the pattern largely held, so the trial is not over.
# Discussion: what further count would change your verdict?


## The statistics of counts

Before comparing counts you need three habits: look at the distribution (not just the
mean), divide by a denominator, and only then ask if a gap is real.


In [ ]:
# Habit 1: look at the distribution. "night" per Dracula paragraph:
night_per = pd.Series([len(re.findall(r"\bnight\b", p.lower())) for p in drac])
print(night_per.describe().round(2))
night_per.hist(bins=range(0, night_per.max() + 2), color="#2e5f8a", figsize=(6, 2.6))
plt.title('"night" per paragraph of Dracula'); plt.xlabel("count in a paragraph")
plt.ylabel("paragraphs"); plt.tight_layout(); plt.show()
# Mean and median disagree because count data is skewed: most paragraphs have zero,
# a few have many. That is the normal shape of counts; a lone mean hides it.


In [ ]:
# Habit 2: divide by a denominator. The novels are longer than all sonnets combined,
# so raw totals always vote for the novels. Rates per 1,000 words compare fairly.
def per_1000(docs_, word):
    hits = sum(len(re.findall(rf"\b{word}\b", d.lower())) for d in docs_)
    return 1000 * hits / sum(len(tokenize(d)) for d in docs_)

for w in ["love", "night", "blood"]:
    print(f"{w:6s}", {name: round(per_1000(dd, w), 2) for name, dd in corpora.items()})
# Habit 3: a gap between rates can still be luck. Whether THIS gap beats chance is
# the shuffle test's question, at the end of the notebook.


## tf-idf: "common here, rare overall"

Raw counts are dominated by words that are common *everywhere* (*the*, *you*).

In [ ]:
docs = poems + frank[:80] + drac[:80]
labels = ["sonnet"] * len(poems) + ["frankenstein"] * 80 + ["dracula"] * 80
vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform(docs)
terms = np.array(vec.get_feature_names_out())

# The most distinctive terms of three famous sonnets, by tf-idf:
for i in [0, 17, 129]:
    row = X[i].toarray().ravel()
    print(f"sonnet {i+1:3d}:", ", ".join(terms[row.argsort()[::-1][:4]]))
# "Common here, rare overall": tf-idf finds what marks a document out from the pile.

## Clustering: three communities, separable from counts alone

For clustering you want a corpus where the groups really differ, so: Reddit. Three
communities with three vocabularies, streamed live from a public research mirror
(analyze-only: we publish counts and findings, never the text, and no usernames).
Ask k-means for three piles without telling it the subreddits, and see if its piles
match.


In [ ]:
from itertools import islice
from datasets import load_dataset

def load_subreddit(sub, n=120):
    """Stream n comments from one subreddit; nothing is saved or republished."""
    ds = load_dataset("parquet", split="train", streaming=True,
        data_files=f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/data/{sub}-*.parquet")
    texts = (r["body"] for r in ds)
    keep = (t for t in texts if isinstance(t, str) and 300 < len(t) < 1500
            and "[removed]" not in t and "[deleted]" not in t)
    return list(islice(keep, n))

SUBS = ["buildapc", "gardening", "relationship_advice"]   # three worlds, three vocabularies
posts, subs_true = [], []
for sub in SUBS:
    got = load_subreddit(sub)
    posts += got; subs_true += [sub] * len(got)
print(len(posts), "comments from", len(SUBS), "subreddits")
print("one from each:")
for sub in SUBS:
    print(f"  {sub:20s}", next(p for p, l in zip(posts, subs_true) if l == sub)[:70], "...")


In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

# Three representations of the same comments, weakest to strongest.
raw_r = CountVectorizer(stop_words="english", min_df=2).fit_transform(posts)
tfidf_r = TfidfVectorizer(stop_words="english", min_df=2).fit_transform(posts)

def cluster_table(M, name):
    piles = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(M)
    print(name)
    print(pd.crosstab(pd.Series(subs_true, name="truth"), pd.Series(piles, name="pile")), "\n")
    return piles

cluster_table(raw_r, "raw counts:")          # one blob: everywhere-words dominate
cluster_table(tfidf_r, "tf-idf:")            # better, still smeared

# Step three: compress the thousands of word-columns into 60 summary dimensions
# (TruncatedSVD is PCA for sparse counts), then cluster. A preview of Week 5's
# whole idea: replace word columns with learned dimensions.
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
Z = normalize(TruncatedSVD(60, random_state=0).fit_transform(tfidf_r))
piles = cluster_table(Z, "tf-idf + 60 dimensions:")
# About six comments in seven land with their community, and no one told k-means the
# labels. The representation did the work: same data, same algorithm, three answers.


In [ ]:
from sklearn.decomposition import PCA

xy = PCA(n_components=2).fit_transform(Z)
colors = {"buildapc": "#2e5f8a", "gardening": "#3f6f5f", "relationship_advice": "#7a3b2e"}
plt.figure(figsize=(6.5, 4.5))
for name, col in colors.items():
    pts = xy[[j for j, l in enumerate(subs_true) if l == name]]
    plt.scatter(pts[:, 0], pts[:, 1], s=14, alpha=0.7, label=name, color=col)
plt.legend(); plt.title("360 comments, placed by their word counts")
plt.tight_layout(); plt.show()
# Three communities, three regions, from counting alone. The stragglers are worth
# reading: a gardening comment about a partner, a buildapc comment about budgets.
# The cluster found vocabulary, and vocabulary mostly tracks community. Which
# difference a cluster found is always your claim to defend, not the algorithm's.


### The hard version: one city, two communities

buildapc against gardening is easy mode: different topics, different words. r/sandiego
and r/SanDiegan are two communities discussing the *same* city. Does counting still
separate them?


In [ ]:
# One request per subreddit to the Arctic Shift archive (the Pushshift successor).
# Analyze-only, as always: counts and findings, never the text or usernames.
import requests

def arctic_sample(sub, n=100):
    r = requests.get("https://arctic-shift.photon-reddit.com/api/comments/search",
                     params={"subreddit": sub, "limit": n, "fields": "body"}, timeout=30)
    return [d["body"] for d in r.json()["data"]
            if isinstance(d.get("body"), str) and 80 < len(d["body"]) < 800
            and "[removed]" not in d["body"] and "[deleted]" not in d["body"]]

sd, sdn = arctic_sample("sandiego"), arctic_sample("SanDiegan")
hard_true = ["sandiego"] * len(sd) + ["SanDiegan"] * len(sdn)
print(len(sd), "r/sandiego,", len(sdn), "r/SanDiegan comments")

Xh = normalize(TruncatedSVD(40, random_state=0).fit_transform(
    TfidfVectorizer(stop_words="english", min_df=2).fit_transform(sd + sdn)))
piles2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xh)
print(pd.crosstab(pd.Series(hard_true, name="truth"), pd.Series(piles2, name="pile")))
# Compare this table to the three-subreddit one above: it barely beats a coin flip.
# Different TOPICS separate easily; two communities on the same topic barely separate
# at all. Week 3 trains a supervised classifier on exactly this pair; even with the
# labels shown, it reaches only about 60 percent, and the words that earn those
# percentage points are the finding.


## Go further: cross-corpus counting (not scheduled in the session)

The corpus choice changes what "common" means. It also works as a corpus sampler before Week 4.

Same counter, three corpora.

In [ ]:
for name, dd in corpora.items():
    text = " ".join(dd)
    top = Counter(t for t in tokenize(text) if t not in STOP).most_common(5)
    print(f"{name:8s}:", ", ".join(f"{w}({n})" for w, n in top))

### A signature vocabulary

Counting alone can show you a *voice*: the words one source uses far more than a baseline.

In [ ]:
# Words the sonnets use much more than the novels, by simple rate ratio.
def rates(text):
    toks = tokenize(text)
    n = len(toks) or 1
    return {w: c / n for w, c in Counter(toks).items()}

sig = rates(" ".join(poems))
base = rates(" ".join(frank[:200]) + " " + " ".join(drac[:200]))
distinctive = sorted(((w, sig[w] / (base.get(w, 0) + 1e-6)) for w, r in sig.items() if sig[w] > 3e-4),
                     key=lambda t: -t[1])[:10]
print("distinctively sonnet-speak:", ", ".join(w for w, _ in distinctive))

## Keyness: distinctively hers, done properly

The ratio above was a rough cut. The standard tool is a **log-odds ratio**: for every word,
how much more likely is it in corpus A than corpus B (with a little smoothing so rare words
don't explode)? Positive means "distinctively A," negative "distinctively B."

**The reveal:** this is *exactly* the method behind Week 1's featured study. *She Giggles, He
Gallops* is a log-odds ratio of stage-direction verbs after "she" vs. "he," run on 2,000
screenplays. You now own the arithmetic behind the piece you admired.

In [ ]:
import numpy as np
from collections import Counter

def counts(texts):
    c = Counter()
    for t in texts:
        c.update(tokenize(t))
    return c

A, B = counts(poems), counts(frank[:200] + drac[:200])   # sonnets vs. the novels
vocab = [w for w in (A | B) if (A[w] + B[w]) >= 3]        # ignore the rarest words
NA, NB = sum(A.values()), sum(B.values())

def log_odds(w):
    pa = (A[w] + 1) / (NA + len(vocab))                   # +1 smoothing
    pb = (B[w] + 1) / (NB + len(vocab))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))

scored = sorted(vocab, key=log_odds)
print("distinctively SONNETS:", [w for w in scored[-8:][::-1]])
print("distinctively NOVELS: ", [w for w in scored[:8]])

## Is the difference real? The shuffle test

Your top keyness word is more common in the poems than in the baseline. But small corpora
produce flukes. The honest check needs no formulas: **shuffle the labels and count again**.
If documents were randomly dealt into "poems" and "baseline" piles a thousand times, how
often would chance alone produce a gap as big as the real one? Rarely: you have a finding.
Often: you have a coincidence. (And a gap can be real *and* tiny; the shuffle test says
"probably not chance," never "big enough to matter." That's your call to argue.)

In [ ]:
rng = np.random.default_rng(0)
word = scored[-1]                                          # our most "distinctively sonnet" word
docs_all = poems + frank[:80] + drac[:80]
word_counts = np.array([tokenize(d).count(word) for d in docs_all])
totals = np.array([max(1, len(tokenize(d))) for d in docs_all])
labels = np.array([1] * len(poems) + [0] * 160)            # 1 = sonnet, 0 = novel paragraph

def rate_gap(lab):
    return word_counts[lab == 1].sum() / totals[lab == 1].sum() - word_counts[lab == 0].sum() / totals[lab == 0].sum()

observed = rate_gap(labels)
shuffled = []
for _ in range(1000):
    rng.shuffle(labels)
    shuffled.append(rate_gap(labels))

beat = sum(abs(g) >= abs(observed) for g in shuffled)
plt.figure(figsize=(6, 3.5))
plt.hist(shuffled, bins=30)
plt.axvline(observed, color="red", linewidth=2, label=f"the real gap for {word!r}")
plt.title(f"1,000 shuffles: chance matched the real gap {beat} times")
plt.xlabel("gap produced by a random shuffle"); plt.ylabel("shuffles")
plt.legend(); plt.tight_layout(); plt.show()
print(f"chance beat the observed gap in {beat}/1000 shuffles",
      "- probably not chance." if beat < 50 else "- could easily be chance; don't lean on it.")

## Where counting fails

Counting is the course's foundation, and it has known holes. Name them now; Week 5
exists to patch some of them.


In [ ]:
line = "I am not happy, and this is not a love story."
print("the counter sees:", [w for w in tokenize(line) if w in ("happy", "love")])
# 1. Negation: happy and love count as present; the sentence denies both.
#    (Bigrams catch a little of this: "not happy" is countable. Most context is not.)

try:
    print("2. stemmer collisions:", stem("universal"), stem("university"), stem("universe"))
    print("3. lemmas need context:", lemma("saw"), "the tool, vs", lemma("saw", "v"), "the past of see")
except NameError:
    print("(run the NLTK cell above to see the stemmer and lemma examples)")

# 4. Polysemy: "sick beat" and "sick child" add to the same count. A word type is not
#    a meaning.
# Every method this week inherits all four holes: tf-idf reweights the counts, k-means
# groups by them, keyness compares them. None of them reads. Week 5's embeddings were
# invented for exactly these failures, and they bring their own.


## Playground: your question

Everything above is a kit. Pick a question and answer it with the pieces: choose any two
corpora (or two slices of one), a word or a pattern, and run keyness plus the shuffle
test on it. Write the question first, in one sentence; the cell is yours to edit.

In [ ]:
# MY QUESTION: ...one sentence here...
GROUP_A = poems               # try: frank, drac, poems[:77], poems[77:]
GROUP_B = drac[:200]          # any other pile of texts

A2, B2 = counts(GROUP_A), counts(GROUP_B)
vocab2 = [w for w in (A2 | B2) if (A2[w] + B2[w]) >= 5]
NA2, NB2 = sum(A2.values()), sum(B2.values())
def lo(w):
    pa = (A2[w] + 1) / (NA2 + len(vocab2)); pb = (B2[w] + 1) / (NB2 + len(vocab2))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))
sc = sorted(vocab2, key=lo)
print("distinctively A:", sc[-8:][::-1])
print("distinctively B:", sc[:8])
# Now shuffle-test your most distinctive word with the cell above: swap it into `word`,
# rebuild docs_all/labels from YOUR two groups, and see whether chance can match the gap.

## You made a counter, and you can defend it

You defined what a word is, counted three corpora three ways, found the words that make a
voice distinctive with the same method as a famous published study, and shuffle-tested the
gap so you know it isn't chance.

**Sketch (homework):** count something in a text you love; make one chart; write one sentence
naming a choice you made. If your count compares two things, shuffle-test the gap.